In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xtrain = np.array(
      [[0.5       , 0.33333333],
       [0.25      , 0.66666667],
       [0.75      , 0.11111111],
       [0.125     , 0.44444444],
       [0.625     , 0.77777778],
       [0.375     , 0.22222222],
       [0.875     , 0.55555556],
       [0.0625    , 0.88888889],
       [0.5625    , 0.03703704],
       [0.3125    , 0.37037037],
       [0.8125    , 0.7037037 ],
       [0.1875    , 0.14814815],
       [0.6875    , 0.48148148],
       [0.4375    , 0.81481481],
       [0.9375    , 0.25925926],
       [0.03125   , 0.59259259],
       [0.53125   , 0.92592593],
       [0.28125   , 0.07407407],
       [0.78125   , 0.40740741],
       [0.15625   , 0.74074074]])

ytrain = np.array(
      [-0.10316614,  0.16585917, -0.29220324,  0.15771768,
        0.06451588, -0.13031895, -0.12112112,  0.43093146,
       -0.34623426, -0.01752642, -0.03618134, -0.11282839,
       -0.08835435,  0.15300116, -0.26063376,  0.45675513,
        0.19451957, -0.23697354, -0.14697874,  0.26922018])

x = np.arange(10)
plt.plot(x, x**2)
plt.show()

## GPySurrogate

In [ ]:
from profit.sur.gaussian_process import GPySurrogate

gp = GPySurrogate()
gp.train(xtrain, ytrain.reshape([-1,1]))

gp.plot()

## Linear regression without transformation

In [ ]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression().fit(xtrain, ytrain)

min_val = xtrain.min(axis=0)
max_val = xtrain.max(axis=0)
npoints = [50] * len(min_val)
xpredict = np.array([np.linspace(minv, maxv, n) for minv, maxv, n in zip(min_val, max_val, npoints)])
Xpredict = np.hstack([xx.flatten().reshape([-1, 1]) for xx in np.meshgrid(*xpredict)])
Ypredict = reg.coef_ @ Xpredict.T + reg.intercept_
Ypredict2 = reg.predict(Xpredict)

ax = plt.axes(projection='3d')
ax.scatter(xtrain[:, 0], xtrain[:, 1], ytrain, color='black', marker='x', alpha=0.8)
ax.plot_trisurf(Xpredict[:, 0], Xpredict[:, 1], Ypredict, color='red', alpha=0.8)
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_zlabel('y')
plt.show()

## Linear Regression with transformation to polynomial basis

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
# from sklearn.compose import TransformedTargetRegressor

# transformer = PolynomialFeatures(3)
# regression = LinearRegression()
# reg = TransformedTargetRegressor(regressor=regression, transformer=transfomr)

poly = PolynomialFeatures(3)
xtrans = poly.fit_transform(xtrain)
Xtrans = poly.fit_transform(Xpredict)
reg = LinearRegression()
reg.fit(xtrans, ytrain)
Ypredict = reg.predict(Xtrans)

# plot
ax = plt.axes(projection='3d')
ax.scatter(xtrain[:, 0], xtrain[:, 1], ytrain, color='black', marker='x', alpha=0.8)
ax.plot_trisurf(Xpredict[:, 0], Xpredict[:, 1], Ypredict, color='red', alpha=0.8)
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_zlabel('y')
plt.show()

## Linear regression with transformation to polynomial basis and Bayesian Ridge

In [ ]:
from sklearn.linear_model import BayesianRidge

poly = PolynomialFeatures(4)
xtrans = poly.fit_transform(xtrain)
Xtrans = poly.fit_transform(Xpredict)
reg = BayesianRidge(tol=1e-6, fit_intercept=False)
# reg.set_params(alpha_init=1., lambda_init=1e1)
reg.fit(xtrans, ytrain)
Ypredict, Ystd = reg.predict(Xtrans, return_std=True)

ax = plt.axes(projection='3d')
ax.scatter(xtrain[:, 0], xtrain[:, 1], ytrain, color='black', marker='x', alpha=0.8)
ax.plot_trisurf(Xpredict[:, 0], Xpredict[:, 1], Ypredict, color='red', alpha=0.8)
ax.plot_trisurf(Xpredict[:, 0], Xpredict[:, 1], Ypredict+Ystd, color='grey', alpha=0.6)
ax.plot_trisurf(Xpredict[:, 0], Xpredict[:, 1], Ypredict-Ystd, color='grey', alpha=0.6)
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_zlabel('y')
plt.show()

In [ ]:
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from scipy.special import binom
import chaospy

def func(x: np.ndarray = None) -> np.ndarray:
    """This is a docstring

    Here are some extra lines for additional explanation on the
    function, its arguments and results

    Args:
        x: a float or array of floats

    Returns:
        Output of the function
    """
    return x + np.cos(10*x)

np.random.seed(100)
n, m = 17, 8
sigma_n = 0.2
distribution = chaospy.Uniform(0, 1)
x_sample = distribution.sample(n, rule="halton")
orthogonal_expansion = chaospy.generate_expansion(m, distribution)
print(orthogonal_expansion.round(2))
# print(orthogonal_expansion)
y_sample = func(x_sample) + np.random.normal(0, sigma_n, n)
approx_model = chaospy.fit_regression(orthogonal_expansion, x_sample, y_sample)

P = orthogonal_expansion(x_sample)
A_inv = sigma_n**2 * np.linalg.inv(P @ P.T)
coeff = 1/sigma_n**2 * A_inv @ P @ y_sample
x = np.linspace(0, 1, 300)
y = func(x)
y_fit = approx_model(x)
cov = orthogonal_expansion(x).T @ A_inv @ orthogonal_expansion(x)
std = np.sqrt(np.diag(cov))

# plot
fig, ax = plt.subplots()
ax.plot(x, y, 'k')
ax.plot(x_sample, y_sample, 'xk')
ax.plot(x, y_fit, 'r')
ax.fill_between(x, y_fit-2*std, y_fit+2*std, color='r', alpha=0.4)
ax.set_ylim([y.min()-0.3, y.max()+0.3])
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
plt.show()
